# CodeQA Mini Attention (Revisi V2) — Perbandingan Terkontrol GRU vs GRU+Attention

**Konteks Proyek:**
- Paper Acuan: *CodeQA: A Question Answering Dataset for Source Code Comprehension* (Liu & Wan, EMNLP 2021).
- Tema Intern: **Reproduce → Rethink → Reinvent**.
- **Dokumentasi Audit & Koreksi dari Run Pilot:**
  1. *Koreksi Sampling:* Run pilot lama keliru mengambil 500 baris pertama (head 0–499) test set yang kebetulan memiliki anomali proporsi Yes/No sebesar 21,2% (sehingga baseline sepele "selalu jawab yes" mengungguli model dengan EM 20,2%). Pada versi revisi ini, subset diambil secara **acak terdistribusi merata dan deterministik** dari seluruh 56.085 train, 7.000 dev, dan 7.000 test set resmi.
  2. *Koreksi Paritas A/B:* Komponen bersama (Embedding, Encoder GRU, Decoder GRU, Output Linear) kini diinisialisasi dengan **nilai bobot awal yang 100% identik**, namun memiliki alokasi memori independen (*disjoint storage*) agar tidak saling mempengaruhi saat ditraining.
  3. *Koreksi Keputusan Acak:* Generator acak untuk batch shuffling dan *teacher forcing* diselaraskan secara deterministik per epoch untuk kedua varian.
  4. *Koreksi Transparansi Ilmiah:* Catatan riwayat proyek ini mengakui bahwa test set pernah diinspeksi pada fase awal. Pada run ini, test set diisolasi ketat dari pembentukan kosakata, penentuan hyperparameter, maupun pemilihan checkpoint.
  5. *Baseline Pembanding:* Ditambahkan baseline sederhana "Always Predict Yes" sebagai pembanding apakah EM model benar-benar melampaui tebakan konstan.

In [1]:
import os, sys, time, json, math, random, copy, re
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# Seed Utama
SEED = 42
def set_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("mini_attention_v2_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device Aktif: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print(f"Output Directory Baru: {OUTPUT_DIR.resolve()}")

Device Aktif: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total VRAM: 3.73 GB
Output Directory Baru: /home/inaka/Codes/LAB_IC/CodeQA/mini_attention_v2_results


## 1. Validasi Integritas Data & Sampling Acak Deterministik

- Membaca ketiga berkas data (`.question`, `.code`, `.answer`) secara penuh.
- Memvalidasi alignment dan memastikan jumlah baris cocok tanpa memakai `zip()` yang dapat memotong data jika panjang tidak sama.
- Melakukan sampling acak deterministik:
  - **10.000 train** dari 56.085 pasangan resmi
  - **500 dev** dari 7.000 pasangan resmi
  - **500 test** dari 7.000 pasangan resmi
- Menyimpan indeks baris asli ke `subset_indices.json` agar dapat direproduksi persis.

In [2]:
def load_and_validate_full_split(data_dir, split_name, expected_count):
    q_file = Path(data_dir) / split_name / f"{split_name}.question"
    c_file = Path(data_dir) / split_name / f"{split_name}.code"
    a_file = Path(data_dir) / split_name / f"{split_name}.answer"

    with open(q_file, "r", encoding="utf-8") as f:
        questions = [line.strip() for line in f]
    with open(c_file, "r", encoding="utf-8") as f:
        codes = [line.strip() for line in f]
    with open(a_file, "r", encoding="utf-8") as f:
        answers = [line.strip() for line in f]

    assert len(questions) == len(codes) == len(answers) == expected_count, (
        f"Mismatch pada {split_name}: {len(questions)} Q, {len(codes)} C, {len(answers)} A (diharapkan {expected_count})"
    )
    return questions, codes, answers

full_train_q, full_train_c, full_train_a = load_and_validate_full_split("data/python", "train", 56085)
full_dev_q, full_dev_c, full_dev_a = load_and_validate_full_split("data/python", "dev", 7000)
full_test_q, full_test_c, full_test_a = load_and_validate_full_split("data/python", "test", 7000)

print(f"Validasi Full Split OK:")
print(f"  - Train: {len(full_train_q)} baris terverifikasi sinkron")
print(f"  - Dev  : {len(full_dev_q)} baris terverifikasi sinkron")
print(f"  - Test : {len(full_test_q)} baris terverifikasi sinkron")

# Sampling Acak Deterministik
rng_sampling = random.Random(SEED)
train_indices = sorted(rng_sampling.sample(range(len(full_train_q)), 10000))
dev_indices = sorted(rng_sampling.sample(range(len(full_dev_q)), 500))
test_indices = sorted(rng_sampling.sample(range(len(full_test_q)), 500))

# Simpan indeks asli untuk transparansi dan audit
subset_indices = {
    "sampling_seed": SEED,
    "train_indices": train_indices,
    "dev_indices": dev_indices,
    "test_indices": test_indices
}
with open(OUTPUT_DIR / "subset_indices.json", "w", encoding="utf-8") as f:
    json.dump(subset_indices, f)

def extract_subset(qs, cs, as_, indices):
    return [{
        "original_id": idx,
        "question": qs[idx],
        "code": cs[idx],
        "answer": as_[idx]
    } for idx in indices]

raw_data = {
    "train": extract_subset(full_train_q, full_train_c, full_train_a, train_indices),
    "dev": extract_subset(full_dev_q, full_dev_c, full_dev_a, dev_indices),
    "test": extract_subset(full_test_q, full_test_c, full_test_a, test_indices)
}

# Analisis Proporsi Yes/No pada Subset Terpilih
def get_yn_distribution(rows):
    total = len(rows)
    yes = sum(1 for r in rows if r["answer"].strip().lower() == "yes")
    no = sum(1 for r in rows if r["answer"].strip().lower() == "no")
    return {"yes": yes, "no": no, "total_yn": yes + no, "total": total, "yn_pct": (yes + no) / total * 100}

train_yn = get_yn_distribution(raw_data["train"])
dev_yn = get_yn_distribution(raw_data["dev"])
test_yn = get_yn_distribution(raw_data["test"])

print(f"\nDistribusi Yes/No pada Subset Acak:")
print(f"  - Train 10k: Yes={train_yn['yes']}, No={train_yn['no']} | Total Y/N={train_yn['total_yn']} ({train_yn['yn_pct']:.2f}%)")
print(f"  - Dev 500  : Yes={dev_yn['yes']}, No={dev_yn['no']} | Total Y/N={dev_yn['total_yn']} ({dev_yn['yn_pct']:.2f}%)")
print(f"  - Test 500 : Yes={test_yn['yes']}, No={test_yn['no']} | Total Y/N={test_yn['total_yn']} ({test_yn['yn_pct']:.2f}%)")

# Overlap kode
train_codes = set(r["code"] for r in raw_data["train"])
dev_codes = set(r["code"] for r in raw_data["dev"])
test_codes = set(r["code"] for r in raw_data["test"])
print(f"\nOverlap kode string:")
print(f"  - Overlap Train & Dev : {len(train_codes.intersection(dev_codes))}")
print(f"  - Overlap Train & Test: {len(train_codes.intersection(test_codes))}")

Validasi Full Split OK:
  - Train: 56085 baris terverifikasi sinkron
  - Dev  : 7000 baris terverifikasi sinkron
  - Test : 7000 baris terverifikasi sinkron

Distribusi Yes/No pada Subset Acak:
  - Train 10k: Yes=69, No=58 | Total Y/N=127 (1.27%)
  - Dev 500  : Yes=7, No=3 | Total Y/N=10 (2.00%)
  - Test 500 : Yes=5, No=3 | Total Y/N=8 (1.60%)

Overlap kode string:
  - Overlap Train & Dev : 1
  - Overlap Train & Test: 0


## 2. Tokenisasi & Pembangunan Kosakata (Murni dari Train)

- **Tokenizer:** `re.findall(r"\w+|[^\w\s]", text.lower())`.
- **Special Tokens:** `PAD=0`, `BOS=1`, `EOS=2`, `UNK=3`, `SEP=4`.
- Kosakata 8.000 kata dibangun **murni dari 10.000 data train final**.
- **Evaluasi OOV:** Murni diukur pada **Dev Set**, tanpa menyentuh Test Set.

In [3]:
PAD, BOS, EOS, UNK, SEP = range(5)
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<unk>", "<sep>"]

def tokenize(text):
    return re.findall(r"\w+|[^\w\s]", text.lower())

train_token_counts = Counter()
for row in raw_data["train"]:
    train_token_counts.update(tokenize(row["question"]))
    train_token_counts.update(tokenize(row["code"]))
    train_token_counts.update(tokenize(row["answer"]))

VOCAB_SIZE = 8000
vocab = SPECIAL_TOKENS + [w for w, _ in train_token_counts.most_common(VOCAB_SIZE - len(SPECIAL_TOKENS)) if w not in SPECIAL_TOKENS]
stoi = {w: i for i, w in enumerate(vocab)}

print(f"Total token unik di Train 10k: {len(train_token_counts):,}")
print(f"Ukuran Kosakata Final       : {len(vocab):,}")

# Evaluasi OOV murni pada 500 Dev
dev_ans_tokens = [tok for row in raw_data["dev"] for tok in tokenize(row["answer"])]
dev_inp_tokens = [tok for row in raw_data["dev"] for tok in tokenize(row["question"]) + tokenize(row["code"])]

dev_ans_oov = sum(1 for t in dev_ans_tokens if t not in stoi)
dev_inp_oov = sum(1 for t in dev_inp_tokens if t not in stoi)

print(f"OOV pada Dev Answers: {dev_ans_oov / len(dev_ans_tokens) * 100:.2f}% ({dev_ans_oov}/{len(dev_ans_tokens)})")
print(f"OOV pada Dev Inputs : {dev_inp_oov / len(dev_inp_tokens) * 100:.2f}% ({dev_inp_oov}/{len(dev_inp_tokens)})")

Total token unik di Train 10k: 24,355
Ukuran Kosakata Final       : 8,000
OOV pada Dev Answers: 6.91% (177/2562)
OOV pada Dev Inputs : 5.74% (1986/34588)


## 3. Prapemrosesan Tensor & Analisis Pemotongan (*Truncation*)

Konfigurasi yang disetujui dipertahankan:
- `MAX_SOURCE = 160`
- `MAX_TARGET = 24` (termasuk token `<bos>` dan `<eos>`)

In [4]:
MAX_SOURCE = 160
MAX_TARGET = 24

def build_tensor_dataset(rows):
    sources, lengths, targets = [], [], []
    truncated_src, truncated_tgt = 0, 0
    
    for r in rows:
        q_toks = [stoi.get(t, UNK) for t in tokenize(r["question"])]
        c_toks = [stoi.get(t, UNK) for t in tokenize(r["code"])]
        a_toks = [stoi.get(t, UNK) for t in tokenize(r["answer"])]
        
        full_src = q_toks + [SEP] + c_toks
        if len(full_src) > MAX_SOURCE:
            truncated_src += 1
        src = full_src[:MAX_SOURCE]
        
        full_tgt = a_toks
        if len(full_tgt) > MAX_TARGET - 2:
            truncated_tgt += 1
        tgt = [BOS] + full_tgt[:MAX_TARGET - 2] + [EOS]
        
        lengths.append(len(src))
        sources.append(src + [PAD] * (MAX_SOURCE - len(src)))
        targets.append(tgt + [PAD] * (MAX_TARGET - len(tgt)))
        
    ds = TensorDataset(torch.tensor(sources, dtype=torch.long),
                       torch.tensor(lengths, dtype=torch.long),
                       torch.tensor(targets, dtype=torch.long))
    trunc_info = {
        "src_trunc_pct": (truncated_src / len(rows)) * 100,
        "tgt_trunc_pct": (truncated_tgt / len(rows)) * 100
    }
    return ds, trunc_info

train_dataset, train_trunc = build_tensor_dataset(raw_data["train"])
dev_dataset, dev_trunc = build_tensor_dataset(raw_data["dev"])
test_dataset, test_trunc = build_tensor_dataset(raw_data["test"])

print(f"Train Tensor: {tuple(train_dataset.tensors[0].shape)} | Trunc Source: {train_trunc['src_trunc_pct']:.2f}% | Trunc Target: {train_trunc['tgt_trunc_pct']:.2f}%")
print(f"Dev Tensor  : {tuple(dev_dataset.tensors[0].shape)} | Trunc Source: {dev_trunc['src_trunc_pct']:.2f}% | Trunc Target: {dev_trunc['tgt_trunc_pct']:.2f}%")
print(f"Test Tensor : {tuple(test_dataset.tensors[0].shape)} (DIKARANTINA)")

Train Tensor: (10000, 160) | Trunc Source: 4.95% | Trunc Target: 0.29%
Dev Tensor  : (500, 160) | Trunc Source: 4.40% | Trunc Target: 0.20%
Test Tensor : (500, 160) (DIKARANTINA)


## 4. Arsitektur Model & Kontrol Paritas A/B yang Ketat

- **Varian A (Baseline Kontrol):** Bidirectional GRU Encoder (hidden 128*2=256) + Unidirectional GRU Decoder (hidden 256) tanpa attention.
- **Varian B (Intervensi):** Encoder dan Decoder dengan dimensi identik, ditambah **Luong General Attention**.
- **Kontrol Paritas Bobot Awal:**
  - Komponen bersama (`embedding`, `encoder GRU`, `decoder GRU`, `output projection`) diinisialisasi dari basis yang sama persis (`copy.deepcopy`).
  - Dilakukan verifikasi eksplisit bahwa bobot awal bernilai identik (`torch.equal`), namun memiliki pointer memori berbeda (`data_ptr().isdisjoint`).

In [5]:
EMBED_DIM = 128
HIDDEN_DIM = 256

class SharedEncoder(nn.Module):
    def __init__(self, embedding, gru):
        super().__init__()
        self.embedding = embedding
        self.gru = gru
        
    def forward(self, source, lengths):
        embeds = self.embedding(source)
        packed = nn.utils.rnn.pack_padded_sequence(embeds, lengths.cpu(), batch_first=True, enforce_sorted=False)
        enc_out, hidden = self.gru(packed)
        enc_out, _ = nn.utils.rnn.pad_packed_sequence(enc_out, batch_first=True, total_length=source.size(1))
        # Gabungkan forward (hidden[0]) dan backward (hidden[1])
        dec_init = torch.cat([hidden[0], hidden[1]], dim=-1).unsqueeze(0)
        return enc_out, dec_init

class DecoderPlain(nn.Module):
    def __init__(self, embedding, gru, fc_out):
        super().__init__()
        self.embedding = embedding
        self.gru = gru
        self.fc_out = fc_out
        
    def step(self, token, hidden, enc_out, mask):
        embed = self.embedding(token)
        output, hidden = self.gru(embed, hidden)
        logits = self.fc_out(output.squeeze(1))
        return logits, hidden, None

class DecoderAttention(nn.Module):
    def __init__(self, embedding, gru, fc_out, w_a, w_c):
        super().__init__()
        self.embedding = embedding
        self.gru = gru
        self.fc_out = fc_out
        self.W_a = w_a
        self.W_c = w_c
        
    def step(self, token, hidden, enc_out, mask):
        embed = self.embedding(token)
        output, hidden = self.gru(embed, hidden)
        
        # Luong General Attention: score = s_t^T * W_a * h_i
        proj_enc = self.W_a(enc_out)
        scores = torch.bmm(output, proj_enc.transpose(1, 2))
        if mask is not None:
            scores = scores.masked_fill(~mask.unsqueeze(1), -1e9)
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights, enc_out)
        
        concat = torch.cat([context, output], dim=-1)
        combined = torch.tanh(self.W_c(concat))
        logits = self.fc_out(combined.squeeze(1))
        return logits, hidden, attn_weights

class Seq2SeqModel(nn.Module):
    def __init__(self, encoder, decoder, pad_idx=PAD, bos_idx=BOS, eos_idx=EOS, sep_idx=SEP):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.pad_idx = pad_idx
        self.bos_idx = bos_idx
        self.eos_idx = eos_idx
        self.sep_idx = sep_idx
        
    def forward(self, source, lengths, target, tf_generator=None, teacher_forcing_ratio=0.5):
        B, max_len = target.shape
        enc_out, dec_hidden = self.encoder(source, lengths)
        mask = (source != self.pad_idx)
        
        logits_list = []
        token = target[:, 0:1] # BOS
        for t in range(1, max_len):
            logits, dec_hidden, _ = self.decoder.step(token, dec_hidden, enc_out, mask)
            logits_list.append(logits.unsqueeze(1))
            if self.training:
                if tf_generator is not None:
                    use_tf = (torch.rand(1, generator=tf_generator).item() < teacher_forcing_ratio)
                else:
                    use_tf = (random.random() < teacher_forcing_ratio)
            else:
                use_tf = False
            token = target[:, t:t+1] if use_tf else logits.argmax(-1, keepdim=True)
        return torch.cat(logits_list, dim=1)
        
    @torch.no_grad()
    def generate(self, source, lengths, max_target=MAX_TARGET):
        self.eval()
        B = source.size(0)
        enc_out, dec_hidden = self.encoder(source, lengths)
        mask = (source != self.pad_idx)
        
        token = torch.full((B, 1), self.bos_idx, dtype=torch.long, device=source.device)
        done = torch.zeros(B, dtype=torch.bool, device=source.device)
        generated = []
        for _ in range(max_target - 1):
            logits, dec_hidden, _ = self.decoder.step(token, dec_hidden, enc_out, mask)
            logits[:, [self.pad_idx, self.bos_idx, self.sep_idx]] = -float("inf")
            token = logits.argmax(-1, keepdim=True)
            token = torch.where(done[:, None], torch.full_like(token, self.eos_idx), token)
            generated.append(token)
            done |= token.squeeze(1).eq(self.eos_idx)
            if done.all():
                break
        return torch.cat(generated, dim=1) if generated else torch.full((B, 1), self.eos_idx, device=source.device)

def create_strictly_controlled_models(vocab_size=len(vocab), embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, init_seed=SEED):
    torch.manual_seed(init_seed)
    
    # 1. Base weights
    base_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD)
    base_enc_gru = nn.GRU(embed_dim, hidden_dim // 2, batch_first=True, bidirectional=True)
    base_dec_gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
    base_fc_out = nn.Linear(hidden_dim, vocab_size)
    
    # 2. Varian A: Salin bobot, pisahkan storage memori
    embed_a = copy.deepcopy(base_embed)
    enc_gru_a = copy.deepcopy(base_enc_gru)
    dec_gru_a = copy.deepcopy(base_dec_gru)
    fc_out_a = copy.deepcopy(base_fc_out)
    
    enc_a = SharedEncoder(embed_a, enc_gru_a)
    dec_a = DecoderPlain(embed_a, dec_gru_a, fc_out_a)
    model_a = Seq2SeqModel(enc_a, dec_a)
    
    # 3. Varian B: Salin bobot dasar yang identik, tambahkan layer atensi terpisah
    embed_b = copy.deepcopy(base_embed)
    enc_gru_b = copy.deepcopy(base_enc_gru)
    dec_gru_b = copy.deepcopy(base_dec_gru)
    fc_out_b = copy.deepcopy(base_fc_out)
    
    w_a = nn.Linear(hidden_dim, hidden_dim, bias=False)
    w_c = nn.Linear(2 * hidden_dim, hidden_dim)
    
    enc_b = SharedEncoder(embed_b, enc_gru_b)
    dec_b = DecoderAttention(embed_b, dec_gru_b, fc_out_b, w_a, w_c)
    model_b = Seq2SeqModel(enc_b, dec_b)
    
    # 4. Verifikasi Paritas Eksplisit
    assert torch.equal(model_a.encoder.embedding.weight, model_b.encoder.embedding.weight), "Bobot Embedding tidak identik!"
    assert torch.equal(model_a.encoder.gru.weight_ih_l0, model_b.encoder.gru.weight_ih_l0), "Bobot Encoder GRU tidak identik!"
    assert torch.equal(model_a.decoder.gru.weight_ih_l0, model_b.decoder.gru.weight_ih_l0), "Bobot Decoder GRU tidak identik!"
    assert torch.equal(model_a.decoder.fc_out.weight, model_b.decoder.fc_out.weight), "Bobot FC Out tidak identik!"
    
    ptrs_a = set(p.data_ptr() for p in model_a.parameters())
    ptrs_b = set(p.data_ptr() for p in model_b.parameters())
    assert ptrs_a.isdisjoint(ptrs_b), "Model A dan B tidak boleh berbagi pointer memori tensor!"
    
    return model_a, model_b

model_a, model_b = create_strictly_controlled_models()
model_a, model_b = model_a.to(DEVICE), model_b.to(DEVICE)

params_a = sum(p.numel() for p in model_a.parameters())
params_b = sum(p.numel() for p in model_b.parameters())

print(f"Kontrol A/B Berhasil Terverifikasi:")
print(f"  - Parameter Varian A (Plain GRU)    : {params_a:,} parameter")
print(f"  - Parameter Varian B (GRU+Attention): {params_b:,} parameter")
print(f"  - Bobot awal shared layers terbukti identik 100% dan memori disjoint.")

Kontrol A/B Berhasil Terverifikasi:
  - Parameter Varian A (Plain GRU)    : 3,574,592 parameter
  - Parameter Varian B (GRU+Attention): 3,771,456 parameter
  - Bobot awal shared layers terbukti identik 100% dan memori disjoint.


## 5. Uji Sanity Check Mekanis (Probe 4 Sampel)

Uji mekanis untuk memastikan:
- Dimensi forward pass tepat.
- Gradien bernilai *finite* (`torch.isfinite`).
- Padding mask pada attention bekerja (bobot PAD < 1e-4 dan softmax menjumlahkan ke 1,0).
- Loss mengabaikan PAD (`ignore_index=PAD`).

In [6]:
probe_loader = DataLoader(train_dataset, batch_size=4, shuffle=False)
probe_src, probe_len, probe_tgt = next(iter(probe_loader))
probe_src, probe_len, probe_tgt = probe_src.to(DEVICE), probe_len.to(DEVICE), probe_tgt.to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=PAD)

# 1. Forward shape check
out_a = model_a(probe_src, probe_len, probe_tgt)
out_b = model_b(probe_src, probe_len, probe_tgt)
assert out_a.shape == (4, MAX_TARGET - 1, len(vocab))
assert out_b.shape == (4, MAX_TARGET - 1, len(vocab))

# 2. Gradient finiteness check
loss_b = criterion(out_b.reshape(-1, len(vocab)), probe_tgt[:, 1:].reshape(-1))
loss_b.backward()
for name, param in model_b.named_parameters():
    if param.grad is not None:
        assert torch.isfinite(param.grad).all(), f"Gradien non-finite pada {name}"
model_b.zero_grad(set_to_none=True)

# 3. Attention sum and pad masking check
with torch.no_grad():
    enc_out, dec_hid = model_b.encoder(probe_src, probe_len)
    mask = (probe_src != PAD)
    tok = probe_tgt[:, 0:1]
    _, _, attn_w = model_b.decoder.step(tok, dec_hid, enc_out, mask)
    attn_sum = attn_w.sum(dim=-1)
    assert torch.allclose(attn_sum, torch.ones_like(attn_sum), atol=1e-4), "Softmax atensi tidak berjumlah 1.0"
    pad_positions = ~mask
    if pad_positions.any():
        assert (attn_w.squeeze(1)[pad_positions] < 1e-4).all(), "Token padding masih menerima atensi signifikan"

# 4. Probe optimization trend check (5 langkah)
opt_probe = torch.optim.Adam(model_b.parameters(), lr=0.01)
initial_loss = float(loss_b)
for _ in range(5):
    opt_probe.zero_grad()
    o = model_b(probe_src, probe_len, probe_tgt, teacher_forcing_ratio=1.0)
    l = criterion(o.reshape(-1, len(vocab)), probe_tgt[:, 1:].reshape(-1))
    l.backward()
    opt_probe.step()
final_loss = float(l)
assert final_loss < initial_loss, f"Loss probe gagal turun: {initial_loss:.4f} -> {final_loss:.4f}"

# Buang seluruh state probe dan buat instance bersih kembali
del model_a, model_b, opt_probe
model_a, model_b = create_strictly_controlled_models()
model_a, model_b = model_a.to(DEVICE), model_b.to(DEVICE)

print("SANITY CHECK MEKANIS LULUS (Model bersih di-instansiasi kembali).")

/tmp/ipykernel_57128/211587892.py:35: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  initial_loss = float(loss_b)


SANITY CHECK MEKANIS LULUS (Model bersih di-instansiasi kembali).


## 6. Benchmark Waktu & Memori GPU (Minibatch Representatif)

Uji 100 minibatch (10 warmup, 90 measured) menggunakan batch nyata dari dataset train untuk mengukur waktu per step dan peak VRAM aktual secara presisi.

In [7]:
BATCH_SIZE = 32

def run_hardware_benchmark(name, model, dataset):
    bench_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                              generator=torch.Generator().manual_seed(SEED))
    bench_iter = iter(bench_loader)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss(ignore_index=PAD)
    
    # 10 batch warmup
    for _ in range(10):
        try:
            src, lens, tgt = next(bench_iter)
        except StopIteration:
            bench_iter = iter(bench_loader)
            src, lens, tgt = next(bench_iter)
        src, lens, tgt = src.to(DEVICE), lens.to(DEVICE), tgt.to(DEVICE)
        opt.zero_grad()
        out = model(src, lens, tgt, teacher_forcing_ratio=0.5)
        loss = crit(out.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats(DEVICE)
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    
    # 90 batch measured
    for _ in range(90):
        try:
            src, lens, tgt = next(bench_iter)
        except StopIteration:
            bench_iter = iter(bench_loader)
            src, lens, tgt = next(bench_iter)
        src, lens, tgt = src.to(DEVICE), lens.to(DEVICE), tgt.to(DEVICE)
        opt.zero_grad()
        out = model(src, lens, tgt, teacher_forcing_ratio=0.5)
        loss = crit(out.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
        loss.backward()
        opt.step()
        
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        peak_vram_mb = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
    else:
        peak_vram_mb = 0.0
        
    elapsed = time.perf_counter() - t0
    step_ms = (elapsed / 90) * 1000
    throughput = (90 * BATCH_SIZE) / elapsed
    epoch_batches = math.ceil(len(raw_data["train"]) / BATCH_SIZE)
    est_train_epoch_sec = (epoch_batches * elapsed) / 90
    
    return {
        "variant": name,
        "step_ms": step_ms,
        "throughput_samples_sec": throughput,
        "est_train_epoch_sec": est_train_epoch_sec,
        "peak_vram_mb": peak_vram_mb
    }

bench_a = run_hardware_benchmark("Varian A (Plain GRU)", model_a, train_dataset)
bench_b = run_hardware_benchmark("Varian B (GRU+Attention)", model_b, train_dataset)

# Reset kondisi bersih sebelum training final
del model_a, model_b
set_all_seeds(SEED)
model_a, model_b = create_strictly_controlled_models()
model_a, model_b = model_a.to(DEVICE), model_b.to(DEVICE)

benchmark_report = {"variant_a": bench_a, "variant_b": bench_b}
with open(OUTPUT_DIR / "benchmark_report.json", "w", encoding="utf-8") as f:
    json.dump(benchmark_report, f, indent=2)

print("=== HASIL BENCHMARK EMPIRIS (MINIBATCH REAL TRAIN) ===")
print(f"Varian A: Step {bench_a['step_ms']:.1f} ms | Est. Train Epoch {bench_a['est_train_epoch_sec']:.1f} s | Peak VRAM {bench_a['peak_vram_mb']:.1f} MB")
print(f"Varian B: Step {bench_b['step_ms']:.1f} ms | Est. Train Epoch {bench_b['est_train_epoch_sec']:.1f} s | Peak VRAM {bench_b['peak_vram_mb']:.1f} MB")

MAX_EPOCHS = 10
PATIENCE = 4
print(f"\nKebijakan Training Pre-Declared:")
print(f"  - Maksimum Epoch: {MAX_EPOCHS}")
print(f"  - Early Stopping Patience: {PATIENCE} epoch")
print(f"  - Checkpoint metric: Dev Generation Token F1 (Dev loss sebagai tie-breaker)")

=== HASIL BENCHMARK EMPIRIS (MINIBATCH REAL TRAIN) ===
Varian A: Step 24.7 ms | Est. Train Epoch 7.7 s | Peak VRAM 244.0 MB
Varian B: Step 57.1 ms | Est. Train Epoch 17.9 s | Peak VRAM 384.5 MB

Kebijakan Training Pre-Declared:
  - Maksimum Epoch: 10
  - Early Stopping Patience: 4 epoch
  - Checkpoint metric: Dev Generation Token F1 (Dev loss sebagai tie-breaker)


## 7. Training & Early Stopping Berbasis Dev F1

- **Pencegahan Overfitting & Pemilihan Checkpoint:** Di setiap akhir epoch, model dievaluasi pada 500 Dev set (Dev Loss dan Dev Generation Token F1).
- Checkpoint terbaik disimpan berdasarkan skor **Dev Token F1** tertinggi (Dev loss sebagai pemecah seri).
- Test set tetap berada dalam karantina total selama proses ini berlangsung.

In [8]:
def decode_tokens(ids):
    tokens = []
    for idx in ids:
        if idx == EOS:
            break
        if idx not in (PAD, BOS, SEP):
            tokens.append(vocab[idx])
    return " ".join(tokens)

def compute_metrics_eval(predictions, references):
    assert len(predictions) == len(references)
    total = len(references)
    em_all, f1_all = 0, []
    em_yn, f1_yn, gold_yn_cnt, gold_yes_cnt, gold_no_cnt = 0, [], 0, 0, 0
    em_non_yn, f1_non_yn, gold_non_yn_cnt = 0, [], 0
    empty_cnt = 0
    
    for p_str, r_str in zip(predictions, references):
        p_clean = p_str.strip()
        r_clean = r_str.strip()
        if not p_clean:
            empty_cnt += 1
        p_toks = tokenize(p_clean)
        r_toks = tokenize(r_clean)
        
        is_em = (p_toks == r_toks)
        if is_em:
            em_all += 1
            
        overlap = sum((Counter(p_toks) & Counter(r_toks)).values())
        if not p_toks and not r_toks:
            f1 = 1.0
        elif not p_toks or not r_toks:
            f1 = 0.0
        else:
            prec = overlap / len(p_toks)
            rec = overlap / len(r_toks)
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        f1_all.append(f1)
        
        is_yes = (r_clean.lower() == "yes")
        is_no = (r_clean.lower() == "no")
        if is_yes or is_no:
            gold_yn_cnt += 1
            if is_yes:
                gold_yes_cnt += 1
            else:
                gold_no_cnt += 1
            if is_em:
                em_yn += 1
            f1_yn.append(f1)
        else:
            gold_non_yn_cnt += 1
            if is_em:
                em_non_yn += 1
            f1_non_yn.append(f1)
            
    p_counts = Counter(predictions)
    top_p, top_cnt = p_counts.most_common(1)[0] if p_counts else ("", 0)
    
    return {
        "overall": {
            "EM": (em_all / total) * 100,
            "token_F1": (sum(f1_all) / total) * 100,
            "em_count": em_all
        },
        "yes_no": {
            "gold_total": gold_yn_cnt,
            "gold_yes": gold_yes_cnt,
            "gold_no": gold_no_cnt,
            "EM": (em_yn / gold_yn_cnt * 100) if gold_yn_cnt > 0 else 0.0,
            "token_F1": (sum(f1_yn) / gold_yn_cnt * 100) if gold_yn_cnt > 0 else 0.0,
            "em_count": em_yn
        },
        "non_yes_no": {
            "gold_total": gold_non_yn_cnt,
            "EM": (em_non_yn / gold_non_yn_cnt * 100) if gold_non_yn_cnt > 0 else 0.0,
            "token_F1": (sum(f1_non_yn) / gold_non_yn_cnt * 100) if gold_non_yn_cnt > 0 else 0.0,
            "em_count": em_non_yn
        },
        "diversity": {
            "unique_predictions": len(p_counts),
            "empty_predictions": empty_cnt,
            "top_1_prediction": top_p,
            "top_1_concentration": (top_cnt / total) * 100
        }
    }

@torch.no_grad()
def evaluate_dev(model, dataset, raw_split_rows):
    model.eval()
    crit = nn.CrossEntropyLoss(ignore_index=PAD)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    loss_sum, token_count = 0.0, 0
    all_preds = []
    
    for src, lens, tgt in loader:
        src, lens, tgt = src.to(DEVICE), lens.to(DEVICE), tgt.to(DEVICE)
        out = model(src, lens, tgt, teacher_forcing_ratio=0.0)
        n = int(tgt[:, 1:].ne(PAD).sum())
        loss = crit(out.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
        loss_sum += float(loss) * n
        token_count += n
        
        gen_ids = model.generate(src, lens, max_target=MAX_TARGET)
        for seq in gen_ids.cpu().tolist():
            all_preds.append(decode_tokens(seq))
            
    dev_loss = loss_sum / token_count
    refs = [r["answer"] for r in raw_split_rows]
    metrics = compute_metrics_eval(all_preds, refs)
    return dev_loss, metrics, all_preds

def train_model_controlled(model_name, model, train_ds, dev_ds, raw_dev_rows, max_epochs=MAX_EPOCHS, patience=PATIENCE):
    print(f"\n==========================================")
    print(f"Memulai Training Terkontrol: {model_name}")
    print(f"==========================================")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD)
    
    best_dev_f1 = -1.0
    best_dev_loss = float("inf")
    best_epoch = 0
    best_state = None
    history = []
    no_improve_epochs = 0
    
    start_time = time.perf_counter()
    for ep in range(max_epochs):
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  generator=torch.Generator().manual_seed(SEED + ep))
        tf_gen = torch.Generator().manual_seed(SEED + ep)
        
        model.train()
        sum_loss, sum_tokens = 0.0, 0
        for src, lens, tgt in train_loader:
            src, lens, tgt = src.to(DEVICE), lens.to(DEVICE), tgt.to(DEVICE)
            optimizer.zero_grad()
            out = model(src, lens, tgt, tf_generator=tf_gen, teacher_forcing_ratio=0.5)
            loss = criterion(out.reshape(-1, len(vocab)), tgt[:, 1:].reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            n = int(tgt[:, 1:].ne(PAD).sum())
            sum_loss += float(loss.detach()) * n
            sum_tokens += n
            
        train_loss = sum_loss / sum_tokens
        dev_loss, dev_met, _ = evaluate_dev(model, dev_ds, raw_dev_rows)
        dev_f1 = dev_met["overall"]["token_F1"]
        dev_em = dev_met["overall"]["EM"]
        
        ep_info = {
            "epoch": ep + 1,
            "train_loss": train_loss,
            "dev_loss": dev_loss,
            "dev_EM": dev_em,
            "dev_token_F1": dev_f1,
            "dev_unique": dev_met["diversity"]["unique_predictions"],
            "dev_top_conc": dev_met["diversity"]["top_1_concentration"]
        }
        history.append(ep_info)
        
        is_best = (dev_f1 > best_dev_f1) or (dev_f1 == best_dev_f1 and dev_loss < best_dev_loss)
        if is_best:
            best_dev_f1 = dev_f1
            best_dev_loss = dev_loss
            best_epoch = ep + 1
            best_state = copy.deepcopy(model.state_dict())
            no_improve_epochs = 0
            marker = " [BEST DEV CHECKPOINT SAVED]"
        else:
            no_improve_epochs += 1
            marker = ""
            
        print(f"Epoch {ep+1:02d}/{max_epochs:02d} | Train Loss: {train_loss:.3f} | Dev Loss: {dev_loss:.3f} | Dev F1: {dev_f1:.2f}% | Dev EM: {dev_em:.1f}%{marker}")
        
        if no_improve_epochs >= patience:
            print(f"Early stopping dipicu pada epoch {ep+1} (tidak ada peningkatan selama {patience} epoch berturut-turut).")
            break
            
    total_time = time.perf_counter() - start_time
    print(f"Training {model_name} Selesai dalam {total_time:.1f} detik ({total_time/60:.2f} menit).")
    print(f"Checkpoint Terpilih: Epoch {best_epoch} (Dev F1: {best_dev_f1:.2f}%)")
    
    model.load_state_dict(best_state)
    return history, total_time, best_epoch

history_a, time_a, best_ep_a = train_model_controlled("Varian A (Plain GRU)", model_a, train_dataset, dev_dataset, raw_data["dev"])
history_b, time_b, best_ep_b = train_model_controlled("Varian B (GRU+Attention)", model_b, train_dataset, dev_dataset, raw_data["dev"])


Memulai Training Terkontrol: Varian A (Plain GRU)


Epoch 01/10 | Train Loss: 5.307 | Dev Loss: 5.218 | Dev F1: 16.30% | Dev EM: 1.6% [BEST DEV CHECKPOINT SAVED]


Epoch 02/10 | Train Loss: 4.617 | Dev Loss: 5.160 | Dev F1: 16.14% | Dev EM: 1.6%


Epoch 03/10 | Train Loss: 4.238 | Dev Loss: 5.165 | Dev F1: 16.46% | Dev EM: 1.6% [BEST DEV CHECKPOINT SAVED]


Epoch 04/10 | Train Loss: 3.871 | Dev Loss: 5.196 | Dev F1: 17.65% | Dev EM: 1.6% [BEST DEV CHECKPOINT SAVED]


...

Epoch 04/10 | Train Loss: 4.269 | Dev Loss: 5.199 | Dev F1: 17.57% | Dev EM: 1.6% [BEST DEV CHECKPOINT SAVED]


Epoch 05/10 | Train Loss: 3.977 | Dev Loss: 5.208 | Dev F1: 16.02% | Dev EM: 1.2%


Epoch 06/10 | Train Loss: 3.680 | Dev Loss: 5.287 | Dev F1: 16.45% | Dev EM: 1.4%


Epoch 07/10 | Train Loss: 3.372 | Dev Loss: 5.365 | Dev F1: 16.42% | Dev EM: 1.2%


Epoch 08/10 | Train Loss: 3.069 | Dev Loss: 5.491 | Dev F1: 16.57% | Dev EM: 1.4%
Early stopping dipicu pada epoch 8 (tidak ada peningkatan selama 4 epoch berturut-turut).
Training Varian B (GRU+Attention) Selesai dalam 152.0 detik (2.53 menit).
Checkpoint Terpilih: Epoch 4 (Dev F1: 17.57%)


## 8. Evaluasi Final pada Test Set Terkarantina & Baseline "Always Yes"

Karantina test set (500 sampel acak) **dibuka tepat satu kali**.
Dievaluasi:
1. **Baseline "Always Yes":** Menjawab "yes" untuk seluruh pertanyaan.
2. **Varian A (Plain GRU):** Checkpoint terbaik dari Dev set.
3. **Varian B (GRU+Attention):** Checkpoint terbaik dari Dev set.

In [9]:
@torch.no_grad()
def evaluate_test_model(model, dataset, raw_split_rows):
    model.eval()
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    all_preds = []
    t0 = time.perf_counter()
    for src, lens, _ in loader:
        src, lens = src.to(DEVICE), lens.to(DEVICE)
        gen_ids = model.generate(src, lens, max_target=MAX_TARGET)
        for seq in gen_ids.cpu().tolist():
            all_preds.append(decode_tokens(seq))
    infer_time = time.perf_counter() - t0
    refs = [r["answer"] for r in raw_split_rows]
    metrics = compute_metrics_eval(all_preds, refs)
    return metrics, all_preds, infer_time

test_metrics_a, test_preds_a, infer_time_a = evaluate_test_model(model_a, test_dataset, raw_data["test"])
test_metrics_b, test_preds_b, infer_time_b = evaluate_test_model(model_b, test_dataset, raw_data["test"])

# Baseline Always Yes
baseline_yes_preds = ["yes"] * len(raw_data["test"])
test_metrics_yes = compute_metrics_eval(baseline_yes_preds, [r["answer"] for r in raw_data["test"]])

print("=== HASIL EVALUASI TEST SET TERKONTROL (500 SAMPEL) ===")
header = f"{'Metrik':<34} | {'Baseline Always Yes':<20} | {'Varian A (Plain)':<18} | {'Varian B (Attention)':<20}"
print(header)
print("-" * len(header))
print(f"{'Overall Exact Match (EM) (%)':<34} | {test_metrics_yes['overall']['EM']:<20.2f} | {test_metrics_a['overall']['EM']:<18.2f} | {test_metrics_b['overall']['EM']:<20.2f}")
print(f"{'Overall Token F1 (%)':<34} | {test_metrics_yes['overall']['token_F1']:<20.2f} | {test_metrics_a['overall']['token_F1']:<18.2f} | {test_metrics_b['overall']['token_F1']:<20.2f}")
print(f"{'Non-Yes/No EM (%)':<34} | {test_metrics_yes['non_yes_no']['EM']:<20.2f} | {test_metrics_a['non_yes_no']['EM']:<18.2f} | {test_metrics_b['non_yes_no']['EM']:<20.2f}")
print(f"{'Non-Yes/No Token F1 (%)':<34} | {test_metrics_yes['non_yes_no']['token_F1']:<20.2f} | {test_metrics_a['non_yes_no']['token_F1']:<18.2f} | {test_metrics_b['non_yes_no']['token_F1']:<20.2f}")
print(f"{'Yes/No EM (%)':<34} | {test_metrics_yes['yes_no']['EM']:<20.2f} | {test_metrics_a['yes_no']['EM']:<18.2f} | {test_metrics_b['yes_no']['EM']:<20.2f}")
print(f"{'Yes/No Token F1 (%)':<34} | {test_metrics_yes['yes_no']['token_F1']:<20.2f} | {test_metrics_a['yes_no']['token_F1']:<18.2f} | {test_metrics_b['yes_no']['token_F1']:<20.2f}")
print(f"{'Total EM Substantif (Non-Y/N)':<34} | {test_metrics_yes['non_yes_no']['em_count']:<20d} | {test_metrics_a['non_yes_no']['em_count']:<18d} | {test_metrics_b['non_yes_no']['em_count']:<20d}")
print(f"{'Total EM Biner (Yes/No)':<34} | {test_metrics_yes['yes_no']['em_count']:<20d} | {test_metrics_a['yes_no']['em_count']:<18d} | {test_metrics_b['yes_no']['em_count']:<20d}")
print(f"{'Jumlah Prediksi Unik':<34} | {test_metrics_yes['diversity']['unique_predictions']:<20d} | {test_metrics_a['diversity']['unique_predictions']:<18d} | {test_metrics_b['diversity']['unique_predictions']:<20d}")
print(f"{'Jawaban Kosong':<34} | {test_metrics_yes['diversity']['empty_predictions']:<20d} | {test_metrics_a['diversity']['empty_predictions']:<18d} | {test_metrics_b['diversity']['empty_predictions']:<20d}")
print(f"{'Top-1 Konsentrasi (%)':<34} | {test_metrics_yes['diversity']['top_1_concentration']:<20.2f} | {test_metrics_a['diversity']['top_1_concentration']:<18.2f} | {test_metrics_b['diversity']['top_1_concentration']:<20.2f}")
print(f"{'Prediksi Terbanyak':<34} | {test_metrics_yes['diversity']['top_1_prediction']:>20s} | {test_metrics_a['diversity']['top_1_prediction']:>18s} | {test_metrics_b['diversity']['top_1_prediction']:>20s}")

=== HASIL EVALUASI TEST SET TERKONTROL (500 SAMPEL) ===
Metrik                             | Baseline Always Yes  | Varian A (Plain)   | Varian B (Attention)
-----------------------------------------------------------------------------------------------------
Overall Exact Match (EM) (%)       | 1.00                 | 1.00               | 1.00                
Overall Token F1 (%)               | 1.00                 | 15.22              | 15.67               
Non-Yes/No EM (%)                  | 0.00                 | 0.00               | 0.00                
Non-Yes/No Token F1 (%)            | 0.00                 | 14.45              | 14.91               
Yes/No EM (%)                      | 62.50                | 62.50              | 62.50               
Yes/No Token F1 (%)                | 62.50                | 62.50              | 62.50               
Total EM Substantif (Non-Y/N)      | 0                    | 0                  | 0                   
Total EM Biner (Yes/No)   

## 9. Analisis Kualitatif & Pola Kesalahan (*Error Analysis*)

- Menampilkan 10 contoh pertama tanpa seleksi.
- Menampilkan contoh Exact Match (jika ada).
- Menampilkan contoh kegagalan beserta analisis pola kesalahannya.

In [10]:
print("=== 10 CONTOH PERTAMA TEST SET TANPA SELEKSI ===")
for i in range(10):
    gold = raw_data["test"][i]["answer"]
    q = raw_data["test"][i]["question"]
    p_a = test_preds_a[i] or "<kosong>"
    p_b = test_preds_b[i] or "<kosong>"
    print(f"\n[Test #{i+1} | ID Asli: {raw_data['test'][i]['original_id']}] Q: {q}")
    print(f"  Gold     : {gold}")
    print(f"  Varian A : {p_a}")
    print(f"  Varian B : {p_b}")

print("\n\n=== CONTOH EXACT MATCH PADA VARIAN B ===")
correct_samples_b = [i for i, (p, r) in enumerate(zip(test_preds_b, [r["answer"] for r in raw_data["test"]]))
                     if tokenize(p) == tokenize(r)]
print(f"Total Exact Match pada Varian B: {len(correct_samples_b)} contoh")
if correct_samples_b:
    for i in correct_samples_b[:5]:
        q = raw_data["test"][i]["question"]
        gold = raw_data["test"][i]["answer"]
        p_b = test_preds_b[i]
        print(f"\n[Test #{i+1}] Q: {q}")
        print(f"  Prediksi B : {p_b} (Gold: {gold})")
else:
    print("Tidak ada contoh exact match yang memenuhi kriteria.")

print("\n\n=== CONTOH KEGAGALAN DAN ANALISIS POLA KESALAHAN ===")
failed_samples_b = [i for i, (p, r) in enumerate(zip(test_preds_b, [r["answer"] for r in raw_data["test"]]))
                    if tokenize(p) != tokenize(r)]
for count, i in enumerate(failed_samples_b[:5]):
    q = raw_data["test"][i]["question"]
    gold = raw_data["test"][i]["answer"]
    p_b = test_preds_b[i] or "<kosong>"
    print(f"\n[Gagal #{count+1} - Test #{i+1}] Q: {q}")
    print(f"  Gold       : {gold}")
    print(f"  Prediksi B : {p_b}")

=== 10 CONTOH PERTAMA TEST SET TANPA SELEKSI ===

[Test #1 | ID Asli: 47] Q: Do the python sandbox test on a production server ?
  Gold     : Yes
  Varian A : no
  Varian B : no

[Test #2 | ID Asli: 53] Q: Are orgs considered within a microsite ?
  Gold     : Yes
  Varian A : no
  Varian B : no

[Test #3 | ID Asli: 64] Q: What exists on the system ?
  Gold     : ipvsadm command
  Varian A : the
  Varian B : the

[Test #4 | ID Asli: 81] Q: Does the code find the name of the zone which contains the specified name ?
  Gold     : Yes
  Varian A : yes
  Varian B : yes

[Test #5 | ID Asli: 108] Q: How does a new clip return a new clip ?
  Gold     : with a mask for transparency
  Varian A : with a <unk>
  Varian B : by the

[Test #6 | ID Asli: 113] Q: What does the code remove for testing ?
  Gold     : original unmodified values
  Varian A : a list of
  Varian B : the

[Test #7 | ID Asli: 130] Q: Can we make only ?
  Gold     : Yes
  Varian A : no
  Varian B : no

[Test #8 | ID Asli: 133] Q

## 10. Penyimpanan Checkpoint, Verifikasi Reload & Ekspor Hasil Final

- Menyimpan bobot dan konfigurasi ke disk.
- Uji reload deterministik via `torch.equal`.
- Ekspor seluruh prediksi ke JSONL dan ringkasan akhir ke `final_results.json`.

In [11]:
ckpt_a_path = OUTPUT_DIR / "variant_a.pt"
ckpt_b_path = OUTPUT_DIR / "variant_b.pt"

torch.save({"model_state": model_a.state_dict(), "vocab": vocab,
            "config": {"embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "vocab_size": len(vocab)}}, ckpt_a_path)
torch.save({"model_state": model_b.state_dict(), "vocab": vocab,
            "config": {"embed_dim": EMBED_DIM, "hidden_dim": HIDDEN_DIM, "vocab_size": len(vocab)}}, ckpt_b_path)

# Verifikasi Reload pada Varian B
reloaded_model_a, reloaded_model_b = create_strictly_controlled_models()
state_b = torch.load(ckpt_b_path, map_location=DEVICE, weights_only=True)
reloaded_model_b.load_state_dict(state_b["model_state"])
reloaded_model_b = reloaded_model_b.to(DEVICE)

test_probe_src = test_dataset.tensors[0][:10].to(DEVICE)
test_probe_len = test_dataset.tensors[1][:10].to(DEVICE)

orig_out = model_b.generate(test_probe_src, test_probe_len)
reloaded_out = reloaded_model_b.generate(test_probe_src, test_probe_len)

assert torch.equal(orig_out, reloaded_out), "Hasil checkpoint yang dimuat ulang tidak identik secara biner!"
print("Verifikasi Reload: CHECKPOINT DETERMINISTIK IDENTIK 100% LULUS.")

def export_jsonl(path, rows, predictions):
    with open(path, "w", encoding="utf-8") as f:
        for r, p in zip(rows, predictions):
            f.write(json.dumps({**r, "prediction": p}, ensure_ascii=False) + "\n")

export_jsonl(OUTPUT_DIR / "predictions_variant_a.jsonl", raw_data["test"], test_preds_a)
export_jsonl(OUTPUT_DIR / "predictions_variant_b.jsonl", raw_data["test"], test_preds_b)
export_jsonl(OUTPUT_DIR / "predictions_baseline_yes.jsonl", raw_data["test"], baseline_yes_preds)

final_summary = {
    "protocol": {
        "version": "v2_strictly_controlled",
        "seed": SEED,
        "train_size": len(raw_data["train"]),
        "dev_size": len(raw_data["dev"]),
        "test_size": len(raw_data["test"]),
        "vocab_size": len(vocab),
        "max_source": MAX_SOURCE,
        "max_target": MAX_TARGET
    },
    "hardware": {
        "device": str(DEVICE),
        "gpu_name": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
    },
    "baseline_always_yes": test_metrics_yes,
    "variant_a_plain": {
        "parameters": params_a,
        "training_time_sec": time_a,
        "inference_time_sec": infer_time_a,
        "selected_epoch": best_ep_a,
        "test_metrics": test_metrics_a
    },
    "variant_b_attention": {
        "parameters": params_b,
        "training_time_sec": time_b,
        "inference_time_sec": infer_time_b,
        "selected_epoch": best_ep_b,
        "test_metrics": test_metrics_b
    }
}

with open(OUTPUT_DIR / "final_results.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2)

print(f"Seluruh berkas eksperimen V2 berhasil disimpan di: {OUTPUT_DIR.resolve()}")
print("Selesai.")

Verifikasi Reload: CHECKPOINT DETERMINISTIK IDENTIK 100% LULUS.


Seluruh berkas eksperimen V2 berhasil disimpan di: /home/inaka/Codes/LAB_IC/CodeQA/mini_attention_v2_results
Selesai.
